In [9]:
from __future__ import annotations
import os
import json
import importlib
import pprint
import time

import random
import numpy as np
np.set_printoptions(4)

import torch

from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

from core import bob_search_v03_0_0

from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k_hf
# from utils.utils import add_completions_to_dataset


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.4
llm_gpu_memory_utilization = 0.7

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    dtype="float16",
    seed=0,
)

print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))


INFO 11-02 11:37:01 [config.py:823] This model supports multiple tasks: {'generate', 'score', 'reward', 'classify', 'embed'}. Defaulting to 'generate'.
WARNING 11-02 11:37:01 [config.py:3271] Casting torch.bfloat16 to torch.float16.
WARNING 11-02 11:37:01 [arg_utils.py:1642] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
WARNING 11-02 11:37:01 [cuda.py:91] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 11-02 11:37:01 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.1) with config: model='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='/groups/kjun/tnn/datasets//Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=5000, download_dir=None, load_format=LoadFormat.AUTO, tensor_par

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 11-02 11:37:04 [default_loader.py:272] Loading weights took 1.27 seconds
INFO 11-02 11:37:05 [model_runner.py:1203] Model loading took 2.3185 GiB and 1.385168 seconds
INFO 11-02 11:37:06 [worker.py:294] Memory profiling takes 0.51 seconds
INFO 11-02 11:37:06 [worker.py:294] the current vLLM instance can use total_gpu_memory (31.73GiB) x gpu_memory_utilization (0.70) = 22.21GiB
INFO 11-02 11:37:06 [worker.py:294] model weights take 2.32GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 1.19GiB; the rest of the memory reserved for KV Cache is 18.62GiB.
INFO 11-02 11:37:06 [executor_base.py:113] # cuda blocks: 38125, # CPU blocks: 32768
INFO 11-02 11:37:06 [executor_base.py:118] Maximum concurrency for 5000 tokens per request: 122.00x
INFO 11-02 11:37:12 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 7.14 seconds
#--- memory: 20.95147705078125


In [5]:
# prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
# print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

In [17]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.bs = 8
config.lookahead = 0
config.max_depths = 40
config.sort_completed = False
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

config.version = "v03_0_0"

In [14]:
# load data
level = 4
num_trials = 2
print(f"num_trials = {num_trials}")

dataset = load_data_prm800k_hf(data_dir, level)

num_questions = len(dataset)
num_questions = 2
print(f"num_questions = {num_questions}")

# get batch of questions
batch_of_questions = [dataset[q_idx]['problem'] for q_idx in range(num_questions)]

# run search_algo and save results
config_name = f"bob--level-{level}--{config.version}--n-{config.n}"
print(config_name)
result_dir = f"results/bon--level-{level}/{config_name}"
try:
    os.makedirs(result_dir)
    print(f"Directory '{result_dir}' created successfully.")
except FileExistsError:
    print(f"Directory '{result_dir}' already exists.")
except OSError as e:
    print(f"Error creating directory: {e}")
    stop

num_trials = 2
num_questions = 2
bon--level-4--v11--n-4
Directory 'results/bon--level-4/bon--level-4--v11--n-4' already exists.


In [16]:
importlib.reload(bob_search_v03_0_0)
start_time1 = time.time()
for trial_idx in range(num_trials):
    print(f"trial {trial_idx}")
    start_time = time.time()
    
    results = bob_search_v03_0_0._search(batch_of_questions, config, trial_idx, llm_vllm)
    with open(f"{result_dir}/generate_{config_name}--trial-{trial_idx:03d}.jsonl", 'w', encoding = 'utf-8') as fout:
        json.dump(results, fout)
        fout.write('\n')

    # compute the time
    if trial_idx % 1 == 0:
        total_time = time.time() - start_time
        # time_per_trial = total_time/(trial_idx+1)
        time_per_trial = total_time
        time_per_question = time_per_trial/num_questions
        # print(f"trial {trial_idx}")
        print(f"it takes {time_per_question:0.4f}s per question")
        print(f"it takes {time_per_trial/3600:0.2f}h per trial")

    # add_completions_to_dataset(result_dir, config_name, dataset, prm, trial_idx, config)

trial 0
it takes 54.1917s per question
it takes 0.03h per trial
trial 1
it takes 29.0662s per question
it takes 0.02h per trial
